In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

In [4]:
def get_gold_data():
    symbol = "GC=F"
    
    print("Downloading gold data...")
    # Downloading data from Yahoo Finance
    df = yf.download(symbol, start="2000-01-01", auto_adjust=True)

    if df.empty:
        print("Data download failed!")
        return

    # Flatten MultiIndex columns if present in newer yfinance versions
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    # ---------------------------
    # 1. BASE INDICATORS CALCULATION
    # ---------------------------
    # Moving Averages
    df['SMA5'] = df['Close'].rolling(5).mean()
    df['SMA20'] = df['Close'].rolling(20).mean()
    df['7d_avg'] = df['Close'].rolling(7).mean()
    df['30d_avg'] = df['Close'].rolling(30).mean()

    # RSI (14 period)
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['rsi'] = 100 - (100 / (1 + rs))

    # Other Features
    df['daily_pct_change'] = df['Close'].pct_change()
    df['volatility_7d'] = df['daily_pct_change'].rolling(7).std()
    df['momentum_14d'] = df['Close'] - df['Close'].shift(14)
    df['price_zscore'] = (df['Close'] - df['30d_avg']) / df['Close'].rolling(30).std()
    df['volume'] = df['Volume']
    df['final_price'] = df['Close']

    # ---------------------------
    # 2. TARGET FEATURE (NO SHIFTING HERE)
    # ---------------------------
    # trend_signal: Agar KAL ka SMA5 > KAL ka SMA20 ho (Future Prediction)
    df['trend_signal'] = (df['SMA5'].shift(-1) > df['SMA20'].shift(-1)).astype(int)

    # ---------------------------
    # 3. FEATURE SHIFTING (To Avoid Data Leakage)
    # ---------------------------
    # Tamam technical features ko 1 din shift kar rahe hain
    # Taake model aaj ka data dekh kar kal ka signal predict kare
    feature_cols = [
        'final_price', 'volume', 'rsi', '7d_avg', '30d_avg', 
        'SMA5', 'SMA20', 'daily_pct_change', 'volatility_7d', 
        'momentum_14d', 'price_zscore'
    ]
    
    # Features ko shift kiya (t-1 data)
    df[feature_cols] = df[feature_cols].shift(1)

    # ---------------------------
    # 4. CLEANUP & SAVE
    # ---------------------------
    # Required columns for final dataset
    required_columns = feature_cols + ['trend_signal']
    
    # Drop rows with NaN (Shifting aur Rolling ki wajah se jo aayi hain)
    df.dropna(inplace=True)
    
    df_final = df[required_columns]

    # Save to CSV file
    file_name = "gold_ml_dataset_2000_2026.csv"
    df_final.to_csv(file_name)

    print("--- Pora Code Update Ho Gaya Hai ---")
    print("Dataset is ready with SMA Crossover & Shifted Features (No Leakage)!")
    print(f"File saved: {file_name}")
    print(f"Total Rows: {len(df_final)}")
    print("\nSample Data (Pichle din ke features + Kal ka target):")
    print(df_final.head())

if __name__ == "__main__":
    get_gold_data()

[*********************100%***********************]  1 of 1 completed

--- Pora Code Update Ho Gaya Hai ---
Dataset is ready with SMA Crossover & Shifted Features (No Leakage)!
File saved: gold_ml_dataset_2000_2026.csv
Total Rows: 6391

Sample Data (Pichle din ke features + Kal ka target):
Price       final_price  volume        rsi      7d_avg     30d_avg  \
Date                                                                 
2000-10-12   270.500000    18.0  50.431059  270.685713  272.929997   
2000-10-13   276.399994     3.0  58.333324  271.371425  273.013330   
2000-10-16   272.399994     0.0  47.098960  271.671426  272.816664   
2000-10-17   271.500000     5.0  46.000014  271.799997  272.633330   
2000-10-18   271.100006     0.0  35.907373  272.057142  272.476664   

Price             SMA5       SMA20  daily_pct_change  volatility_7d  \
Date                                                                  
2000-10-12  270.579999  272.129997         -0.006975       0.005695   
2000-10-13  271.739996  272.329997          0.021811       0.010062   
2000-